<a href="https://colab.research.google.com/github/KoeJuntz/ResearchProject/blob/main/02_distribution_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 3 — Historical quotes and outcome definitions
Run **Runtime → Run all** in this standalone notebook. You do not need to rerun notebook 01. Save this file to your private ResearchProject repository afterward.

We check eight scheduled meetings across the available history. The selection is based on dates, not Treasury outcomes. This is validation, not a regression or a completed implied distribution.

Three separate problems are measured: whether historical quotes exist, whether quotes are aligned to the pre-announcement cutoff, and whether outcome categories identify policy changes precisely. A fresh candle is not proof that the underlying quote was recently updated. Open-ended outcomes remain open-ended.

Sources:
- [Kalshi historical candles](https://docs.kalshi.com/api-reference/historical/get-historical-market-candlesticks)
- [Kalshi historical markets](https://docs.kalshi.com/api-reference/historical/get-historical-markets)
- [Official FOMC calendar](https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm)

No raw response files are written or uploaded automatically. API responses remain in runtime memory; only displayed audit outputs are saved in the notebook.


### 1. Connect and select the validation meetings
Each cutoff below is 16:00 Eastern on the preceding business day. Dates were checked against the official calendar. We use January 31, 2024; the separate January 26 market identifier is not counted as another scheduled meeting.


In [1]:
import time
import re
import math
import requests
import pandas as pd
from IPython.display import display

BASE = "https://external-api.kalshi.com/trade-api/v2"
session = requests.Session()
request_log = []
response_cache = {}

def get_json(path, params=None):
    params = dict(params or {})
    cache_key = (path, tuple(sorted(params.items())))
    if cache_key in response_cache:
        return response_cache[cache_key]
    for attempt in range(3):
        r = session.get(BASE + path, params=params, timeout=30)
        if r.status_code == 429 or r.status_code >= 500:
            if attempt < 2:
                try:
                    delay = float(r.headers.get("Retry-After", 2 ** (attempt + 1)))
                except ValueError:
                    delay = 2 ** (attempt + 1)
                time.sleep(min(max(delay, 1), 30))
                continue
        r.raise_for_status()
        result = r.json()
        request_log.append({
            "path": path, "params": params,
            "retrieved_utc": pd.Timestamp.now(tz="UTC").isoformat()
        })
        response_cache[cache_key] = result
        return result

# Explicit dates avoid treating contract closing times as official release times.
meeting_specs = [
    ("FEDDECISION-23MAY",   "2023-05-03", "2023-05-02"),
    ("FEDDECISION-23SEP",   "2023-09-20", "2023-09-19"),
    ("FEDDECISION-24JAN31", "2024-01-31", "2024-01-30"),
    ("FEDDECISION-24JUN",   "2024-06-12", "2024-06-11"),
    ("FEDDECISION-24SEP",   "2024-09-18", "2024-09-17"),
    ("KXFEDDECISION-24DEC", "2024-12-18", "2024-12-17"),
    ("KXFEDDECISION-25SEP", "2025-09-17", "2025-09-16"),
    ("KXFEDDECISION-26JUL", "2026-07-29", "2026-07-28"),
]
meetings = pd.DataFrame(meeting_specs, columns=["event", "meeting_date", "cutoff_date"])
meetings["cutoff_utc"] = meetings["cutoff_date"].map(
    lambda date: pd.Timestamp(date + " 16:00", tz="America/New_York").tz_convert("UTC")
)
assert meetings["meeting_date"].is_unique
assert len(meetings) == 8
display(meetings)


,event,meeting_date,cutoff_date,cutoff_utc
0,FEDDECISION-23MAY,2023-05-03,2023-05-02,2023-05-02 20:00:00+00:00
1,FEDDECISION-23SEP,2023-09-20,2023-09-19,2023-09-19 20:00:00+00:00
2,FEDDECISION-24JAN31,2024-01-31,2024-01-30,2024-01-30 21:00:00+00:00
3,FEDDECISION-24JUN,2024-06-12,2024-06-11,2024-06-11 20:00:00+00:00
4,FEDDECISION-24SEP,2024-09-18,2024-09-17,2024-09-17 20:00:00+00:00
5,KXFEDDECISION-24DEC,2024-12-18,2024-12-17,2024-12-17 21:00:00+00:00
6,KXFEDDECISION-25SEP,2025-09-17,2025-09-16,2025-09-16 20:00:00+00:00
7,KXFEDDECISION-26JUL,2026-07-29,2026-07-28,2026-07-28 20:00:00+00:00


### 2. Retrieve definitions
Both API tiers are searched for market metadata. Missing events are retained in the final summary. The code reads strike definitions and retains full settlement rules; it does not assign an arbitrary value to an open-ended bucket.


In [2]:
market_rows = []
for path in ["/historical/markets", "/markets"]:
    cursor = None
    seen = set()
    for page in range(20):
        params = {"series_ticker": "KXFEDDECISION", "limit": 100}
        if cursor:
            params["cursor"] = cursor
        result = get_json(path, params)
        market_rows.extend(result["markets"])
        cursor = result.get("cursor")
        if not cursor:
            break
        if cursor in seen:
            raise RuntimeError("Repeated market cursor: inventory incomplete")
        seen.add(cursor)
    assert not cursor, f"Inventory incomplete for {path}"

inventory = pd.DataFrame(market_rows).drop_duplicates("ticker")
selected = inventory.loc[
    inventory["event_ticker"].isin(meetings["event"])
].copy()

def describe_strike(market):
    custom = market.get("custom_strike")
    if isinstance(custom, dict) and len(custom) == 1:
        direction, value = next(iter(custom.items()))
        label = f"{direction} {value}"
    else:
        label = str(market.get("yes_sub_title") or "")
    open_ended = bool(re.search(r">|<|at least|at most|or more|or less", label, re.I))
    exact_bps = float("nan")
    number = re.search(r"-?\d+(?:\.\d+)?", label)
    if not open_ended and number:
        magnitude = abs(float(number.group()))
        if re.search(r"cut", label, re.I):
            exact_bps = -magnitude
        elif re.search(r"hike|increase", label, re.I):
            exact_bps = magnitude
    if not open_ended and re.search(r"no change|no cut/hike|maintains|hold", label, re.I):
        exact_bps = 0.0
    return pd.Series({
        "strike_description": label,
        "open_ended_bucket": open_ended,
        "exact_change_bps": exact_bps,
        "requires_rule_review": open_ended or pd.isna(exact_bps)
    })

strike_details = selected.apply(describe_strike, axis=1)
selected = pd.concat([selected, strike_details], axis=1)
display(selected.reindex(columns=[
    "event_ticker", "ticker", "yes_sub_title", "strike_description",
    "open_ended_bucket", "exact_change_bps", "requires_rule_review"
]).sort_values(["event_ticker", "ticker"]))

print("Full rules for all sampled open-ended contracts:")
for _, market in selected.loc[selected["open_ended_bucket"]].iterrows():
    print("\n", market["ticker"])
    print("Primary:", market.get("rules_primary"))
    print("Secondary:", market.get("rules_secondary"))

print("\nEvents absent from inventory:",
      sorted(set(meetings["event"]) - set(selected["event_ticker"])))


,event_ticker,ticker,yes_sub_title,strike_description,open_ended_bucket,exact_change_bps,requires_rule_review
139,FEDDECISION-23MAY,FEDDECISION-23MAY-C25,Cut 25bps,Cut 25,False,-25.0,False
137,FEDDECISION-23MAY,FEDDECISION-23MAY-H0,No change,Hike 0,False,0.0,False
136,FEDDECISION-23MAY,FEDDECISION-23MAY-H25,Hike 25bps,Hike 25,False,25.0,False
135,FEDDECISION-23MAY,FEDDECISION-23MAY-H50,Hike 50bps,Hike 50,False,50.0,False
138,FEDDECISION-23MAY,FEDDECISION-23MAY-TC25,Cut >25bps,Cut >25,True,NaN,True
134,FEDDECISION-23MAY,FEDDECISION-23MAY-TH50,Hike >50bps,Hike >50bps,True,NaN,True
123,FEDDECISION-23SEP,FEDDECISION-23SEP-C25,Cut 25bps,Cut 25,False,-25.0,False
122,FEDDECISION-23SEP,FEDDECISION-23SEP-C>25,Cut >25bps,Cut >25,True,NaN,True
121,FEDDECISION-23SEP,FEDDECISION-23SEP-H0,Hike 0bps,Hike 0,False,0.0,False
120,FEDDECISION-23SEP,FEDDECISION-23SEP-H25,Hike 25bps,Hike 25,False,25.0,False


Full rules for all sampled open-ended contracts:

 KXFEDDECISION-26JUL-H26
Primary: If the Federal Reserve does a Hike of   25bps on July 29, 2026, then the market resolves to Yes.
Secondary: This market is mutually exclusive. Therefore, if the Federal Reserve hikes by 50bps, the 50bps market will resolve to Yes and the 25bps market will resolve to No. Only one bucket, at maximum, can resolve to Yes. Note 4/28/25: For the markets beginning after the May meeting, if a scheduled FOMC meeting is canceled and does not occur on its scheduled date, then the strike for "Fed maintains rate" will resolve to Yes and all others will resolve to No.

 KXFEDDECISION-26JUL-C26
Primary: If the Federal Reserve does a Cut of   25bps on July 29, 2026, then the market resolves to Yes.
Secondary: This market is mutually exclusive. Therefore, if the Federal Reserve hikes by 50bps, the 50bps market will resolve to Yes and the 25bps market will resolve to No. Only one bucket, at maximum, can resolve to Yes. N

### 3. Retrieve quote observations
First request one-minute candles ending in the last 15 minutes before cutoff. If no candle has both quote sides, inspect seven days of hourly candles. The fallback's older timestamp remains visible.

Bid and ask are YES prices in dollars per $1 payoff. Zero bids and $1 asks are retained; these are potentially uninformative quotes, not missing values. Crossed/invalid quotes are flagged, not replaced with nicer earlier quotes. We retain the most recent candle having both sides and its original values. Quote-update age is unknown: period-end age is labeled separately.

This may take several minutes. Progress prints one line per contract. Per-contract request errors stay in the audit.


In [3]:
def dollar_close(candle, side):
    value = (candle.get(side) or {}).get("close")
    return float(value) if value is not None else float("nan")

def quote_candidates(candles, cutoff):
    candidates = []
    for candle in candles:
        end = pd.to_datetime(candle["end_period_ts"], unit="s", utc=True)
        if end > cutoff:
            continue
        bid = dollar_close(candle, "yes_bid")
        ask = dollar_close(candle, "yes_ask")
        if pd.notna(bid) and pd.notna(ask):
            candidates.append((end, bid, ask, candle))
    return candidates

archive = get_json("/historical/cutoff")
market_boundary = pd.Timestamp(archive["market_settled_ts"]).tz_convert("UTC")

# All requested dates predate the current market archive boundary.
# Only historical markets are used in this specific validation exercise.
assert meetings["cutoff_utc"].lt(market_boundary - pd.Timedelta(days=2)).all(), (
    "Sample is too close to the archive boundary; inspect market routing."
)

quote_rows = []
for _, meeting in meetings.iterrows():
    contracts = selected.loc[selected["event_ticker"].eq(meeting["event"])]
    cutoff = meeting["cutoff_utc"]
    for _, market in contracts.iterrows():
        ticker = market["ticker"]
        row = {
            "event": meeting["event"], "meeting_date": meeting["meeting_date"],
            "contract": ticker, "outcome": market.get("yes_sub_title"),
            "cutoff_utc": cutoff, "quote_period_end_utc": pd.NaT,
            "bid": float("nan"), "ask": float("nan"), "midpoint": float("nan"),
            "spread": float("nan"), "period_age_hours": float("nan"),
            "quote_update_age": "unknown",
            "quote_carried_from_earlier_period": None,
            "open_ended_bucket": bool(market["open_ended_bucket"]),
            "exact_change_bps": market["exact_change_bps"],
            "normalization": "none", "status": "pending"
        }
        try:
            path = f"/historical/markets/{ticker}/candlesticks"
            candles = get_json(path, {
                "start_ts": int((cutoff - pd.Timedelta(minutes=15)).timestamp()),
                "end_ts": int(cutoff.timestamp()),
                "period_interval": 1
            })["candlesticks"]
            candidates = quote_candidates(candles, cutoff)
            interval = 1
            if not candidates:
                candles = get_json(path, {
                    "start_ts": int((cutoff - pd.Timedelta(days=7)).timestamp()),
                    "end_ts": int(cutoff.timestamp()),
                    "period_interval": 60
                })["candlesticks"]
                candidates = quote_candidates(candles, cutoff)
                interval = 60
            row["candles_in_selected_query"] = len(candles)
            row["period_interval_minutes"] = interval
            if not candidates:
                row["status"] = "no two-sided candle in bounded searches"
            else:
                end, bid, ask, candle = max(candidates, key=lambda x: x[0])
                valid = (
                    math.isfinite(bid) and math.isfinite(ask)
                    and 0 <= bid <= ask <= 1
                )
                row.update({
                    "quote_period_end_utc": end,
                    "bid": bid, "ask": ask,
                    "midpoint": (bid + ask) / 2 if valid else float("nan"),
                    "spread": ask - bid,
                    "period_age_hours": (cutoff - end).total_seconds() / 3600,
                    "quote_carried_from_earlier_period": end < cutoff,
                    "period_volume": (
                        float(candle["volume"]) if candle.get("volume") is not None
                        else float("nan")
                    ),
                    "trade_close_same_period": dollar_close(candle, "price"),
                    "status": "valid quote" if valid else "invalid/crossed quote"
                })
        except (requests.RequestException, ValueError, KeyError, TypeError) as exc:
            row["status"] = f"ERROR: {type(exc).__name__}: {exc}"
        quote_rows.append(row)
        print(f"{ticker}: {row['status']}", flush=True)

quote_audit = pd.DataFrame(quote_rows)
display(quote_audit)
assert quote_audit["contract"].is_unique
assert quote_audit.loc[
    quote_audit["quote_period_end_utc"].notna(), "period_age_hours"
].ge(0).all()


FEDDECISION-23MAY-TH50: no two-sided candle in bounded searches
FEDDECISION-23MAY-H50: valid quote
FEDDECISION-23MAY-H25: valid quote
FEDDECISION-23MAY-H0: valid quote
FEDDECISION-23MAY-TC25: valid quote
FEDDECISION-23MAY-C25: no two-sided candle in bounded searches
FEDDECISION-23SEP-H>25: valid quote
FEDDECISION-23SEP-H25: valid quote
FEDDECISION-23SEP-H0: valid quote
FEDDECISION-23SEP-C>25: valid quote
FEDDECISION-23SEP-C25: valid quote
FEDDECISION-24JAN31-H26: no two-sided candle in bounded searches
FEDDECISION-24JAN31-H25: valid quote
FEDDECISION-24JAN31-H0: valid quote
FEDDECISION-24JAN31-C25: valid quote
FEDDECISION-24JAN31-C26: valid quote
FEDDECISION-24JUN-H25: valid quote
FEDDECISION-24JUN-H0: valid quote
FEDDECISION-24JUN-C25: valid quote
FEDDECISION-24JUN-C26: valid quote
FEDDECISION-24SEP-H26: valid quote
FEDDECISION-24SEP-H25: valid quote
FEDDECISION-24SEP-H0: valid quote
FEDDECISION-24SEP-C26: valid quote
FEDDECISION-24SEP-C25: valid quote
KXFEDDECISION-24DEC-H26: valid q

,event,meeting_date,contract,outcome,cutoff_utc,quote_period_end_utc,bid,ask,midpoint,spread,...,quote_update_age,quote_carried_from_earlier_period,open_ended_bucket,exact_change_bps,normalization,status,candles_in_selected_query,period_interval_minutes,period_volume,trade_close_same_period
0,FEDDECISION-23MAY,2023-05-03,FEDDECISION-23MAY-TH50,Hike >50bps,2023-05-02 20:00:00+00:00,NaT,NaN,NaN,NaN,NaN,...,unknown,None,True,NaN,none,no two-sided candle in bounded searches,0,60,NaN,NaN
1,FEDDECISION-23MAY,2023-05-03,FEDDECISION-23MAY-H50,Hike 50bps,2023-05-02 20:00:00+00:00,2023-05-02 16:00:00+00:00,0.00,0.01,0.005,0.01,...,unknown,True,False,50.0,none,valid quote,1,60,467.00,0.01
2,FEDDECISION-23MAY,2023-05-03,FEDDECISION-23MAY-H25,Hike 25bps,2023-05-02 20:00:00+00:00,2023-05-02 19:56:00+00:00,0.92,0.93,0.925,0.01,...,unknown,True,False,25.0,none,valid quote,1,1,8.00,0.93
3,FEDDECISION-23MAY,2023-05-03,FEDDECISION-23MAY-H0,No change,2023-05-02 20:00:00+00:00,2023-05-02 20:00:00+00:00,0.05,0.06,0.055,0.01,...,unknown,False,False,0.0,none,valid quote,59,60,207.00,0.06
4,FEDDECISION-23MAY,2023-05-03,FEDDECISION-23MAY-TC25,Cut >25bps,2023-05-02 20:00:00+00:00,2023-04-29 18:00:00+00:00,0.00,0.01,0.005,0.01,...,unknown,True,True,NaN,none,valid quote,8,60,0.00,NaN
5,FEDDECISION-23MAY,2023-05-03,FEDDECISION-23MAY-C25,Cut 25bps,2023-05-02 20:00:00+00:00,NaT,NaN,NaN,NaN,NaN,...,unknown,None,False,-25.0,none,no two-sided candle in bounded searches,0,60,NaN,NaN
6,FEDDECISION-23SEP,2023-09-20,FEDDECISION-23SEP-H>25,Hike >25bps,2023-09-19 20:00:00+00:00,2023-09-15 13:00:00+00:00,0.00,0.01,0.005,0.01,...,unknown,True,True,NaN,none,valid quote,1,60,50.00,0.01
7,FEDDECISION-23SEP,2023-09-20,FEDDECISION-23SEP-H25,Hike 25bps,2023-09-19 20:00:00+00:00,2023-09-19 17:00:00+00:00,0.01,0.02,0.015,0.01,...,unknown,True,False,25.0,none,valid quote,34,60,504.00,0.01
8,FEDDECISION-23SEP,2023-09-20,FEDDECISION-23SEP-H0,Hike 0bps,2023-09-19 20:00:00+00:00,2023-09-19 17:00:00+00:00,0.98,0.99,0.985,0.01,...,unknown,True,False,0.0,none,valid quote,27,60,0.00,NaN
9,FEDDECISION-23SEP,2023-09-20,FEDDECISION-23SEP-C>25,Cut >25bps,2023-09-19 20:00:00+00:00,2023-09-15 13:00:00+00:00,0.00,0.01,0.005,0.01,...,unknown,True,True,NaN,none,valid quote,1,60,50.00,0.01


### 4. Summarize the evidence
The bid/ask sum check asks whether probabilities summing to one could lie inside all quoted intervals, **conditional on the categories being mutually exclusive and exhaustive and the quotes representing a common time**. It is not a calibration test. A midpoint is a candidate price proxy; no corrected distribution is constructed here.

All-at-cutoff means reported candle periods end at the cutoff. This still does not identify quote-update age. Wide spreads and boundary quotes remain visible. Even complete coherent probabilities do not identify a policy-decision standard deviation if a material category contains several possible decision sizes.


In [4]:
summary_rows = []
for _, meeting in meetings.iterrows():
    event = meeting["event"]
    expected = selected.loc[selected["event_ticker"].eq(event)]
    group = quote_audit.loc[quote_audit["event"].eq(event)]
    good = group.loc[group["status"].eq("valid quote")]
    complete = len(expected) > 0 and len(good) == len(expected)
    at_cutoff = complete and good["quote_period_end_utc"].eq(meeting["cutoff_utc"]).all()
    sum_bids = good["bid"].sum(min_count=1)
    sum_asks = good["ask"].sum(min_count=1)
    midpoint_sum = good["midpoint"].sum(min_count=1)
    row = {
        "event": event,
        "meeting_date": meeting["meeting_date"],
        "expected_contracts": len(expected),
        "valid_quotes": len(good),
        "all_contracts_quoted": complete,
        "all_periods_end_at_cutoff": at_cutoff,
        "sum_bids": sum_bids,
        "sum_asks": sum_asks,
        "sum_midpoints": midpoint_sum,
        "simplex_intersection_if_exhaustive": (
            bool(sum_bids <= 1 + 1e-9 and sum_asks >= 1 - 1e-9)
            if at_cutoff else None
        ),
        "max_spread": good["spread"].max(),
        "oldest_period_hours": good["period_age_hours"].max(),
        "open_ended_categories": int(expected["open_ended_bucket"].sum()),
        "all_bucket_values_exact": (
            len(expected) > 0 and expected["exact_change_bps"].notna().all()
        ),
        "quoted_tail_ask_sum": (
            good.loc[good["open_ended_bucket"], "ask"].sum(min_count=1)
        )
    }
    for hours in [1, 6, 24, 48]:
        row[f"quote_periods_within_{hours}h"] = int(
            good["period_age_hours"].le(hours).sum()
        )
    summary_rows.append(row)

quote_summary = pd.DataFrame(summary_rows)
display(quote_summary)
print("Sampled meetings:", len(meetings))
print("Meetings with all quote periods ending at cutoff:",
      int(quote_summary["all_periods_end_at_cutoff"].sum()))
print("Meetings whose categories all have exact bps values:",
      int(quote_summary["all_bucket_values_exact"].sum()))
print("API requests this run:", len(request_log))
print("\nNo missing prices were set to zero; no prices were normalized.")
print("Open-ended buckets are unresolved; sigma has not been calculated.")
print("Next: review quote coverage and identify a defensible tail treatment or narrow the research design.")
print("Save this executed notebook to ResearchProject / main as 02_distribution_validation.ipynb.")


,event,meeting_date,expected_contracts,valid_quotes,all_contracts_quoted,all_periods_end_at_cutoff,sum_bids,sum_asks,sum_midpoints,simplex_intersection_if_exhaustive,max_spread,oldest_period_hours,open_ended_categories,all_bucket_values_exact,quoted_tail_ask_sum,quote_periods_within_1h,quote_periods_within_6h,quote_periods_within_24h,quote_periods_within_48h
0,FEDDECISION-23MAY,2023-05-03,6,4,False,False,0.97,1.01,0.990,None,0.01,74.000000,2,False,0.01,2,3,3,3
1,FEDDECISION-23SEP,2023-09-20,5,5,True,False,0.99,1.04,1.015,None,0.01,103.000000,2,False,0.02,0,2,2,2
2,FEDDECISION-24JAN31,2024-01-31,5,4,False,False,0.99,1.03,1.010,None,0.01,76.000000,2,False,0.01,0,0,1,3
3,FEDDECISION-24JUN,2024-06-12,4,4,True,False,0.98,1.02,1.000,None,0.01,2.000000,1,False,0.01,2,4,4,4
4,FEDDECISION-24SEP,2024-09-18,5,5,True,False,0.99,1.06,1.025,None,0.03,43.000000,2,False,0.60,3,3,4,5
5,KXFEDDECISION-24DEC,2024-12-18,5,5,True,False,0.99,1.04,1.015,None,0.01,134.000000,2,False,0.02,3,3,4,4
6,KXFEDDECISION-25SEP,2025-09-17,5,5,True,False,0.99,1.04,1.015,None,0.01,0.050000,2,False,0.06,5,5,5,5
7,KXFEDDECISION-26JUL,2026-07-29,5,5,True,False,0.99,1.04,1.015,None,0.01,0.033333,2,False,0.02,5,5,5,5


Sampled meetings: 8
Meetings with all quote periods ending at cutoff: 0
Meetings whose categories all have exact bps values: 0
API requests this run: 69

No missing prices were set to zero; no prices were normalized.
Open-ended buckets are unresolved; sigma has not been calculated.
Next: review quote coverage and identify a defensible tail treatment or narrow the research design.
Save this executed notebook to ResearchProject / main as 02_distribution_validation.ipynb.
